In [1]:
# Load required packages

if(!require("pacman")) install.packages("pacman")
if(!require("foreach")) install.packages("foreach")
if(!require("parallel")) install.packages("parallel")
if(!require("data.table")) install.packages("data.table")
if(!require("plyr")) install.packages("plyr")
if(!require("dplyr")) install.packages("dplyr")
if(!require("docstring")) install.packages("docstring")
if(!require("decoupleR")) install.packages("decoupleR")
if(!require("this.path")) install.packages("this.path")
if(!require("ggpubr")) install.packages("ggpubr")
if(!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
if(!require("viper")) BiocManager::install("viper")

suppressPackageStartupMessages(library(pacman))
suppressPackageStartupMessages(p_load(foreach))
suppressPackageStartupMessages(p_load(parallel))
suppressPackageStartupMessages(p_load(data.table))
suppressPackageStartupMessages(p_load(plyr))
suppressPackageStartupMessages(p_load(dplyr))
suppressPackageStartupMessages(p_load(docstring))
suppressPackageStartupMessages(p_load(decoupleR))
suppressPackageStartupMessages(p_load(this.path))
suppressPackageStartupMessages(p_load(ggpubr))
suppressPackageStartupMessages(p_load(viper))

Loading required package: pacman

Loading required package: foreach

Loading required package: parallel

Loading required package: data.table

Loading required package: plyr

Loading required package: dplyr


Attaching package: 'dplyr'


The following objects are masked from 'package:plyr':

    arrange, count, desc, failwith, id, mutate, rename, summarise,
    summarize


The following objects are masked from 'package:data.table':

    between, first, last


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Loading required package: docstring


Attaching package: 'docstring'


The following object is masked from 'package:utils':

    ?


Loading required package: decoupleR


Attaching package: 'decoupleR'


The following object is masked from 'package:data.table':

    :=


Loading required package: this.path


Attaching package: 'this.path'


The following object i

In [2]:
# Initialization parallelism
# num_cores <- detectCores()
# if (!is.na(num_cores) && (num_cores > 1)) {
#     suppressPackageStartupMessages(p_load("doMC"))
#     cat(paste("Registering ", num_cores - 1, " cores.\n", sep = ""))
#     registerDoMC(cores = (num_cores - 1))
# }

# This code Detects available CPU cores and registers doMC to run foreach tasks in parallel using all but one core.

# Heads up: doMC is not available on Windows, so this code will only work on Unix-like systems (Linux, macOS).

getwd()

[1] "c:/Users/dwool/OneDrive - Fred Hutch Cancer Center/Graduate School/UWT-MSCSS/TCSS-588 Bioinformatics/Team1_FinalProject/morphic-mini-challenge/R"

In [3]:
# Initialize variables
# Copy scRNA-seq metadata and data files from Drive to resources folder.
# root <- dirname(dirname(this.path::this.path())) # This line throws the following error: Error in .in_shell_path(verbose, original, for.msg, contents): R is running from a shell and argument 'FILE' is missing. This is because this.path() is designed to work in interactive R sessions and may not function correctly in certain environments, such as when running scripts or in non-interactive contexts. To fix this, you can set the root directory manually or use an alternative method to determine the file paths.

# So instead, let's try setting the root directory manually. The root of the project is the parent directory of the R folder, which is the current working directory when running this script. So we can set root to the parent of the current working directory.
root <- dirname(getwd()) # Set root to the parent directory of the current working directory, which should be the root of the project.
print(paste("Root directory set to:", root))


resources.dir <- paste0(root, "/resources/") # Adjusted to look for resources folder one level up from the current working directory, since it's a sibling to the R folder, not inside it.
sc.metadata.file <- paste0(resources.dir, "CHOOSE-sc-wt-and-tf-metadata.csv")
sc.data.file <- paste0(resources.dir, "CHOOSE-sc-wt-and-tf-log-norm.csv.gz")
tigress.res.file <- paste0(resources.dir, "postprocessed-tigress_all-tfs-celltypes.csv.gz")
tfs.to.analyze.file <- paste0(resources.dir, "CHOOSE-tf-to-analyze-metadata.csv")

plot.dir <- paste0(root, "/plots/")
dir.create(plot.dir, showWarnings = FALSE)

[1] "Root directory set to: c:/Users/dwool/OneDrive - Fred Hutch Cancer Center/Graduate School/UWT-MSCSS/TCSS-588 Bioinformatics/Team1_FinalProject/morphic-mini-challenge"


In [4]:
# Load in predictions to be evaluated. Here from tigress. Also, move column 6 (cell.type) to column 1.
tigress.res <- as.data.frame(fread(tigress.res.file))
# tigress.res <- tigress.res[, c(6, 1:5, 7:ncol(tigress.res))]
head(tigress.res)
unique(tigress.res$cell.type)

,cell.type,TF,target,importance,cor.p,score,pval,padj,significant
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>
1,ctx.ip,MYT1L,AL627309.1,0.410000000,0.90268146,0.410000000,0.90268146,0.96512090,FALSE
2,ctx.ip,MYT1L,LINC00115,0.009950249,0.00846958,0.009950249,0.00846958,0.06560787,FALSE
3,ctx.ip,MYT1L,NOC2L,0.255000000,0.05270406,0.255000000,0.05270406,0.22797051,FALSE
4,ctx.ip,MYT1L,KLHL17,0.005000000,0.93375723,0.005000000,0.93375723,0.97609606,FALSE
5,ctx.ip,MYT1L,AL645608.7,0.149253731,0.68035340,0.149253731,0.68035340,0.86285540,FALSE
6,ctx.ip,MYT1L,HES4,0.140000000,0.60826767,0.140000000,0.60826767,0.82324124,FALSE


[1] "ctx.ip"  "ctx.npc" "ge.in"

In [5]:
# Read in CHOOSE scRNA-seq data and metadata
fread_with_rownames <- function(file) {
    tbl <- as.data.frame(fread(file))
    rownames(tbl) <- tbl$V1
    tbl <- tbl[, !(colnames(tbl) == "V1")]
    tbl
}

sc.data <- fread_with_rownames(sc.data.file)
sc.metadata <- fread_with_rownames(sc.metadata.file)



# So at present, these CHOOSE scRNA-seq data and metadata files don't yet exist. I need to find out if they're output from tigress or if they're separate files that I need to attain somehow. # UPDATE: I found them. Ka Yee made them available on Google Drive. I've downloaded them to the Data directory within the "Resources" directory for the time being.

In [6]:
dim(sc.metadata)

colnames(sc.metadata)

head(sc.metadata[,1:10])

[1] 7338   75

[1] "orig.ident"                        "nCount_RNA"                       
 [3] "nFeature_RNA"                      "percent.mito"                     
 [5] "percent.ribo"                      "percent.AC.GenBank"               
 [7] "percent.AL.EMBL"                   "percent.LINC"                     
 [9] "percent.MALAT1"                    "percent.antisense"                
[11] "log10.HGA_Markers"                 "log10.nCount_RNA"                 
[13] "log10.nFeature_RNAs"               "library"                          
[15] "integrated_snn_res.0.1"            "integrated_snn_res.0.3"           
[17] "integrated_snn_res.0.5"            "integrated_snn_res.0.7"           
[19] "seurat_clusters"                   "integrated_snn_res.0.3.ordered"   
[21] "integrated_snn_res.0.5.ordered"    "S.Score"                          
[23] "G2M.Score"                         "Phase"                            
[25] "integrated_snn_res.0.3.GO.process" "cl.names.top.gene.res.0.3"        
[27] "cl.names.KnownMarkers.0.3"         "cl.names.top.gene.res.0.5"        
[29] "cl.names.KnownMarkers.0.5"         "gRNA"                             
[31] "celltype_jf"                       "celltype_cl"                      
[33] "state"                             "lineage"                          
[35] "RNA_snn_res.1"                     "RNA_snn_res.2"                    
[37] "celltype_jf_refined"               "clusters"                         
[39] "celltype_cl_refined"               "clusters_ge"                      
[41] "pseudotime"                        "pseudotime_ranks"                 
[43] "celltype_cl_coarse"                "nCount_modules_pos"               
[45] "nFeature_modules_pos"              "nCount_modules_neg"               
[47] "nFeature_modules_neg"              "celltype_cl_coarse2"              
[49] "test_celltype"                     "percent_mito"                     
[51] "percent_ribo"                      "RNA_snn_res.0.8"                  
[53] "sample"                            "SEQID"                            
[55] "CBC"                               "induced"                          
[57] "batch"                             "nCount_integrated"                
[59] "nFeature_integrated"               "nCount_guide_assignments"         
[61] "nFeature_guide_assignments"        "nCount_guide_assignments_ctrl"    
[63] "nFeature_guide_assignments_ctrl"   "transfer_score"                   
[65] "RNA_snn_res.1.5"                   "celltype_jf_ctrl"                 
[67] "RNA_snn_res.10"                    "celltype_ctrl_transfer"           
[69] "this"                              "celltype_ctrl_transfer_score"     
[71] "SEQID_GEX"                         "batch2_pool"                      
[73] "gRNA_assign"                       "gRNA_perturb"                     
[75] "celltype_ctrl_transfer_coarse"

,orig.ident,nCount_RNA,nFeature_RNA,percent.mito,percent.ribo,percent.AC.GenBank,percent.AL.EMBL,percent.LINC,percent.MALAT1,percent.antisense
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
AAACCCAGTTCATCGA-1_1,137116,6954,3747,0.04328444,0.01121657,0.02300834,0.010928962,0.008196721,0.009059534,0.009922347
AAACGCTTCAGTCACA-1_1,137116,33398,6683,0.01305467,0.01949218,0.01982155,0.006018324,0.008383736,0.027426792,0.011228217
AAAGAACCACTACGGC-1_1,137116,4380,2221,0.01438356,0.05639269,0.01803653,0.003881279,0.005707763,0.110730594,0.013926941
AAAGAACTCGAGTGGA-1_1,137116,7696,3195,0.05132536,0.06535863,0.01234407,0.004937630,0.004028067,0.008316008,0.005587318
AAAGGGCTCTCCTGCA-1_1,137116,5630,2517,0.02611012,0.06429840,0.02113677,0.003374778,0.003019538,0.074955595,0.009946714
AAAGTCCCAATTGTGC-1_1,137116,5442,2521,0.04593899,0.05071665,0.01764057,0.011944138,0.005696435,0.064865858,0.005512679


In [7]:
# Read in the TF / cell type combinations to analyze
tfs.to.analyze <- read.csv(tfs.to.analyze.file, header=TRUE)

In [8]:
score.viper <- function(predictions.tbl, sc.metadata, sc.data, 
                        sc.cell.type.col = "celltype_jf", sc.genotype.col = "gRNA_perturb", sc.wt.status = "CTRL") {
    
  #' Apply VIPER to compute single-cell transcription factor (TF) activities from predicted TF targets.
  #' 
  #' @description Compute VIPER TF activity scores using predicted TF targets and wild type (WT) and TF knockout (KO) scRNA-seq
  #' data.
  #' 
  #' @param predictions.tbl data.frame. A data.frame of cell type-specific TF targets, where each row corresponds to a 
  #' cell type-specific TF target, which is described by columns TF, cell.type, target, and score. TF and target are
  #' gene symbols, appearing as row names in the scRNA-seq data in sc.data. score is the strength of the interaction
  #' and should be in [-1, +1], with positive values corresponding to a positively regulated (induced) target and
  #' negative values corresponding to a negatively regulated (repressed) target.
  #' @param sc.metadata data.frame. Metadata describing the single-cell data, where each row is a cell. Row names
  #' correspond to the cell id. The cell's type is provided in column sc.cell.type.col and the KO'ed TF is in
  #' column sc.genotype.col. WT cells have gRNA_perturb set to CTRL.
  #' @param sc.data data.frame. scRNA-seq data, with row names gene symbols and column names cell ids. The data
  #' are log normalized.
  #' @param sc.cell.type.col. character string providing the name of the column in sc.metadata holding the cell type.
  #' @param sc.genotype.col. character string providing the name of the column in sc.metadata holding WT vs TF KO status.
  #' @param sc.wt.status. character string providing the status of WT cells in the sc.genotype.col
  #' @return A data.frame with VIPER TF activity scores for each cell. Each row is a cell, with VIPER TF activity score in column score.
  #' The TF is in column TF. The cell's type is in column cell.type and its WT or TF KO status
  #' is in column condition (as WT or KO, respectively). 
  
  ddply(predictions.tbl,
        .variables = c("TF", "cell.type"),
        .parallel = FALSE,
        .fun = function(df) {
          tf <- df[1, "TF"]
          ct <- df[1, "cell.type"]
          ko.flag <- (sc.metadata[, sc.genotype.col] == tf) & (sc.metadata[, sc.cell.type.col] == ct)
          wt.flag <- (sc.metadata[, sc.genotype.col] == sc.wt.status) & (sc.metadata[, sc.cell.type.col] == ct)
          ko.mat <- sc.data[, ko.flag]
          wt.mat <- sc.data[, wt.flag]
          network <- data.frame("source" = as.character(tf), "target" = as.character(df$target), "mor" = df$score)
          # VIPER seems to crash if we don't have at least two TFs, so create a dummy.
          dummy.network <- network
          dummy.network$source <- "dummy"
          network <- rbind(network, dummy.network)
          combined.mat <- cbind(ko.mat, wt.mat)
          vp <- run_viper(combined.mat, network, .source = "source", .target = "target", .mor = "mor", pleiotropy = FALSE, method = "none")
          vp <- subset(vp, source != "dummy")
          vp <- merge(vp, sc.metadata[ko.flag | wt.flag, c(sc.genotype.col), drop=FALSE], by.x = c("condition"), by.y = c("row.names"))
          vp <- vp[, !(colnames(vp) %in% c("condition", "statistic", "source", "p_value"))]
          colnames(vp)[colnames(vp) == sc.genotype.col] <- "perturbation"
          vp$condition <- vp$perturbation
          vp$perturbation[vp$perturbation == sc.wt.status] <- "none"
          vp$condition[vp$condition != sc.wt.status] <- "KO"
          vp$condition[vp$condition == sc.wt.status] <- "WT"
          vp <- cbind(vp, TF = tf, cell.type = ct)
          vp
        })
}

In [9]:
compare.viper.distributions <- function(viper.df) {
        #' Compare VIPER TF activity scores between WT and TK KO cells.
        #' 
        #' @description Apply a one-sided Wilcoxon test of the hypothesis that TF activity scores are higher in WT than in TF KO cells.
        #' 
        #' @param viper.df data.frame. A data.frame with VIPER TF activity scores for each cell, e.g., as returned by score.viper.
        #' Each row is a cell, with VIPER TF activity score in column score. The TF is in column TF.
        #' The cell's type is in column cell.type and its WT or TF KO status is in column condition (as WT or KO, respectively).
        #' @return A data.frame with the Wilcoxon significance (column pval) that the VIPER activity score is higher for WT
        #' cells than for TF KO cells for the corresponding cell.type and TF.
        stats <- ddply(viper.df, .variables = c("cell.type", "TF"),
        .fun = function(df) {
                ret <- wilcox.test(x = subset(df, condition == "WT")$score,
                y = subset(df, condition == "KO")$score, alternative="greater")
                data.frame(pval = ret$p.value)
                })
        stats <- stats[order(stats$pval, decreasing=FALSE),]
        stats
}

In [10]:
# Normalize TIGRESS cell-type labels (e.g., ctx.ip -> ctx_ip) to match tfs.to.analyze
tigress.res$cell.type <- gsub("\\.", "_", as.character(tigress.res$cell.type))

tmp <- tfs.to.analyze[, c("TF", "celltype_jf")]
colnames(tmp) <- c("TF", "cell.type")
tmp <- merge(tigress.res, tmp)
tg.preds <- subset(tmp, significant == TRUE)

cat("Merged rows:", nrow(tmp), "\n")
cat("Rows in tg.preds:", nrow(tg.preds), "\n")

Merged rows: 86317 
Rows in tg.preds: 9301 


In [11]:
# Compute VIPER TF activity scores for predictions using WT and TF KO scRNA-seq data
tg.viper <- score.viper(tg.preds, sc.metadata, sc.data)

dim(tg.viper)
tg.viper

[1] 1208    5

score,perturbation,condition,TF,cell.type
<dbl>,<chr>,<chr>,<chr>,<chr>
9.243851,none,WT,ASH1L,ctx_ip
9.190868,ASH1L,KO,ASH1L,ctx_ip
20.526005,none,WT,ASH1L,ctx_ip
9.328975,none,WT,ASH1L,ctx_ip
14.149138,none,WT,ASH1L,ctx_ip
12.908425,ASH1L,KO,ASH1L,ctx_ip
9.034172,none,WT,ASH1L,ctx_ip
17.609405,ASH1L,KO,ASH1L,ctx_ip
21.840129,none,WT,ASH1L,ctx_ip


In [12]:
# Compare the distribution (over single cells) of TF activity scores, expecting that 
# TF activity will be higher in WT cells than in KO cells where the TF is disrupted.
tg.viper.pvals <- compare.viper.distributions(tg.viper)
print(tg.viper.pvals)

  cell.type     TF       pval
6     ge_in  MYT1L 0.03464450
2   ctx_npc BCL11A 0.05151462
5     ge_in  KDM5B 0.80215835
1    ctx_ip  ASH1L 0.84554982
4     ge_in  ASH1L 0.90750573
3   ctx_npc  KMT2A 0.97589449


In [13]:
# Create a plot of the distributions of TF activity scores, stratified by WT or KO condition.
# g <- ggplot(data = tg.viper, aes(x = condition, y = score)) + geom_violin()
# Use ggpubr to include stats on plot
g <- ggviolin(data = tg.viper, "condition", "score") + stat_compare_means(method.args = list(alternative = "greater"), ref.group = "KO")
g <- g + facet_wrap(cell.type ~ TF, scales = "free_y")

In [14]:
ofile <- paste0(plot.dir, "tigress-CHOOSE-TF-viper-distributions.png")
png(ofile)
print(g)
d <- dev.off()
print(ofile)

[1] "c:/Users/dwool/OneDrive - Fred Hutch Cancer Center/Graduate School/UWT-MSCSS/TCSS-588 Bioinformatics/Team1_FinalProject/morphic-mini-challenge/plots/tigress-CHOOSE-TF-viper-distributions.png"


In [15]:
# Print the head of the VIPER inputs' data frames to check that they look correct and that the VIPER inputs are being constructed as expected.

print("Head of tg.preds, which represents our input from tigress to VIPER")
head(tg.preds)
dim(tg.preds)
table(tg.preds$TF)
table(tfs.to.analyze$TF)

print("Head of sc.data")
head(sc.data)

print("Head of sc.metadata")
head(sc.metadata)



[1] "Head of tg.preds, which represents our input from tigress to VIPER"


,cell.type,TF,target,importance,cor.p,score,pval,padj,significant
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>
46,ctx_ip,ASH1L,CLK2,0.265,1.420407e-04,0.265,1.420407e-04,2.407567e-03,TRUE
48,ctx_ip,ASH1L,FDPS,0.025,4.399617e-03,0.025,4.399617e-03,3.999805e-02,TRUE
50,ctx_ip,ASH1L,ASH1L,0.000,0.000000e+00,0.000,0.000000e+00,0.000000e+00,TRUE
80,ctx_ip,ASH1L,C1orf159,0.145,2.552993e-04,0.145,2.552993e-04,3.968552e-03,TRUE
82,ctx_ip,ASH1L,KIF1B,0.570,4.110287e-08,0.570,4.110287e-08,1.597503e-06,TRUE
105,ctx_ip,ASH1L,MAN1A2,0.275,9.995065e-04,0.275,9.995065e-04,1.231206e-02,TRUE


[1] 9301    9


 ASH1L BCL11A  KDM5B  KMT2A  MYT1L 
  1603   2046   1121    418   4113 


 ASH1L BCL11A  KDM5B  KMT2A  MYT1L 
     2      1      1      1      1 

[1] "Head of sc.data"


,AAACCCAGTTCATCGA-1_1,AAACGCTTCAGTCACA-1_1,AAAGAACCACTACGGC-1_1,AAAGAACTCGAGTGGA-1_1,AAAGGGCTCTCCTGCA-1_1,AAAGTCCCAATTGTGC-1_1,AAAGTGACAAATACAG-1_1,AACAAAGGTCGGAAAC-1_1,AACAACCTCCATCTCG-1_1,AACACACGTTGAGAGC-1_1,⋯,TGGGCTGTCCCGAACG-1_8,TGGTTAGAGCTTTGTG-1_8,TGTTCATGTGGGCTCT-1_8,TTAATCCAGCGCCTAC-1_8,TTACGTTGTGCCTGAC-1_8,TTCCAATAGATCACTC-1_8,TTCTTCCAGGTTCACT-1_8,TTCTTCCCATTGACTG-1_8,TTTCAGTTCCCGTGTT-1_8,TTTGGAGAGCATAGGC-1_8
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
AL627309.1,0,0.2629286,0,0.0000000,0.000000,0,0.000000,0.0000000,0,0,⋯,0,0,0.000000,0,0.0000000,0,0,0,0,0
LINC00115,0,0.0000000,0,0.0000000,1.023704,0,0.000000,0.0000000,0,0,⋯,0,0,0.000000,0,0.0000000,0,0,0,0,0
NOC2L,0,0.0000000,0,0.8345517,0.000000,0,0.701757,0.0000000,0,0,⋯,0,0,0.000000,0,0.4639345,0,0,0,0,0
KLHL17,0,0.0000000,0,0.0000000,0.000000,0,0.000000,0.0000000,0,0,⋯,0,0,0.000000,0,0.0000000,0,0,0,0,0
AL645608.7,0,0.0000000,0,0.0000000,0.000000,0,0.000000,0.0000000,0,0,⋯,0,0,0.654651,0,0.4639345,0,0,0,0,0
HES4,0,0.0000000,0,0.0000000,0.000000,0,0.000000,0.7084404,0,0,⋯,0,0,0.000000,0,0.4639345,0,0,0,0,0


[1] "Head of sc.metadata"


,orig.ident,nCount_RNA,nFeature_RNA,percent.mito,percent.ribo,percent.AC.GenBank,percent.AL.EMBL,percent.LINC,percent.MALAT1,percent.antisense,⋯,celltype_jf_ctrl,RNA_snn_res.10,celltype_ctrl_transfer,this,celltype_ctrl_transfer_score,SEQID_GEX,batch2_pool,gRNA_assign,gRNA_perturb,celltype_ctrl_transfer_coarse
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<chr>,<int>,<chr>,<lgl>,<dbl>,<lgl>,<lgl>,<chr>,<chr>,<chr>
AAACCCAGTTCATCGA-1_1,137116,6954,3747,0.04328444,0.01121657,0.02300834,0.010928962,0.008196721,0.009059534,0.009922347,⋯,NA,90,L6_CThPN,FALSE,1.0000000,NA,NA,MECP2,MECP2,L6_CThPN
AAACGCTTCAGTCACA-1_1,137116,33398,6683,0.01305467,0.01949218,0.01982155,0.006018324,0.008383736,0.027426792,0.011228217,⋯,NA,85,L56,FALSE,0.5842084,NA,NA,FOXP1,FOXP1,L56
AAAGAACCACTACGGC-1_1,137116,4380,2221,0.01438356,0.05639269,0.01803653,0.003881279,0.005707763,0.110730594,0.013926941,⋯,NA,3,CGE_IN,FALSE,0.8609244,NA,NA,KDM5B,KDM5B,CGE_IN
AAAGAACTCGAGTGGA-1_1,137116,7696,3195,0.05132536,0.06535863,0.01234407,0.004937630,0.004028067,0.008316008,0.005587318,⋯,NA,34,L23,FALSE,1.0000000,NA,NA,FOXP1,FOXP1,L23
AAAGGGCTCTCCTGCA-1_1,137116,5630,2517,0.02611012,0.06429840,0.02113677,0.003374778,0.003019538,0.074955595,0.009946714,⋯,NA,39,CGE_LGE_IN,FALSE,0.7754296,NA,NA,MYT1L,MYT1L,CGE_LGE_IN
AAAGTCCCAATTGTGC-1_1,137116,5442,2521,0.04593899,0.05071665,0.01764057,0.011944138,0.005696435,0.064865858,0.005512679,⋯,NA,48,IP,FALSE,0.9610091,NA,NA,ASH1L,ASH1L,IP
